# Load Data

In [1]:
import pandas as pd
import numpy as np
import datetime
pd.set_option('display.max_columns', None)

In [2]:
ev = pd.read_excel('../../processed_data/02_matched_with_pagetimes/evaluator_full_merged.xlsx')

In [3]:
ev['female'] = np.nan
ev.loc[ev['Sex']=='Female', 'female'] = 1
ev.loc[ev['Sex']=='Male', 'female'] = 0

#calculate weighted average
ev['wa_difficulty'] = np.nan
for i in range(ev.shape[0]):
    wa = 0
    for j in range(11):
        wa += (ev['difficulty' + str(j)][i]/100)*j
    ev.loc[i, 'wa_difficulty'] = wa
    
#calculate median difficulty
median_variable_list = []
for index, row in ev[['difficulty0', 'difficulty1', 'difficulty2', 'difficulty3', 'difficulty4', 'difficulty5', 'difficulty6', 'difficulty7', 'difficulty8', 'difficulty9', 'difficulty10']].iterrows():
    cumulative_sum = 0
    for column, value in row.iteritems():
        cumulative_sum += value
        if cumulative_sum >= 50:
            median_variable_list.append(column)
            break
ev['med_difficulty'] = pd.Series(median_variable_list)
ev['med_difficulty'] = [int(''.join(filter(str.isdigit, element))) for element in ev['med_difficulty']]

#attention
ev['pass_att2'] = 0
ev.loc[ev['attention']==0.5, 'pass_att2'] = 1

#create variable for people that never punish
ev['never_punish'] = 0
ev.loc[(ev['punish_del_good'] == 0) & (ev['punish_del_bad'] == 0) & (ev['punish_nodel_bad'] == 0) & (ev['punish_nodel_good'] == 0), 'never_punish'] = 1

#punishment differences
ev['bad_good_del'] = ev['punish_del_bad'] - ev['punish_del_good']
ev['bad_good_nodel'] = ev['punish_nodel_bad'] - ev['punish_nodel_good']
ev['nodel_del_good'] = ev['punish_nodel_good'] - ev['punish_del_good']
ev['nodel_del_bad'] = ev['punish_nodel_bad'] - ev['punish_del_bad']

#frequency of punishment variable
for i in ['punish_del_good', 'punish_del_bad', 'punish_nodel_good', 'punish_nodel_bad']:
    ev[i+'_bin'] = 0
    ev.loc[ev[i]>0, i+'_bin'] = 1

#convert risk task switching point to interval
ev.loc[ev['switching_point']==9999, 'switching_point'] = 110

#for each person calculate how much time they spent on the punishment screen
ev['p_time'] = np.nan
for i in range(ev.shape[0]):
    ev.loc[i, 'p_time'] = (datetime.datetime.utcfromtimestamp(ev['PBE'][i]) - datetime.datetime.utcfromtimestamp(ev['P'][i])).total_seconds()
#one person in pilot had different order
ev.loc[ev['participant.code']=='snj20i5j', 'p_time'] = (datetime.datetime.utcfromtimestamp(ev['Risk'][i]) - datetime.datetime.utcfromtimestamp(ev['P'][i])).total_seconds()

#socio-demographics cleaning
ev['white'] = 1
ev.loc[ev['Ethnicity']!='White/Caucasian', 'white']=0

ev['tech_use_at_work_often'] = 0
ev.loc[ev['Technology use at work']=='more than once a day', 'tech_use_at_work_often'] = 1

ev['leader'] = 0
ev.loc[(ev['Management experience']=="Yes") & (ev['Leadership/position of power/supervisory duties']=="Yes"), 'leader']=1

ev['went_to_uni'] = ev['Highest education level completed'].apply(
    lambda x: 1 if x in [
        'Undergraduate degree (BA/BSc/other)',
        'Graduate degree (MA/MSc/MPhil/other)',
        'Doctorate degree (PhD/other)'
    ] else 0
)

ev['religious'] = 1
ev.loc[ev['Religious affiliation']=="Non Religious (e.g. Agnostic, Atheist, No Religion)", 'religious'] = 0

ev['technology_score'] = 0
# Increase the technology score by 1 for each condition met
ev['technology_score'] += ev['Weekly device usage'].apply(lambda x: 1 if x == 'Multiple times every day' else 0)
ev['technology_score'] += ev['Dating apps'].apply(lambda x: 1 if x == 'Yes' else 0)
ev['technology_score'] += ev['Computer programming'].apply(lambda x: 1 if x == 'Yes' else 0)
ev['technology_score'] += ev['Cryptocurrency'].apply(lambda x: 1 if x == 'Yes' else 0)


C:\Users\Felix\AppData\Local\Temp\ipykernel_22644\3721304507.py:17: FutureWarning: iteritems is deprecated and will be removed in a future version. Use .items instead.
  for column, value in row.iteritems():


In [4]:
rename_dict = {
    'participant.code': 'code',
    'session.code': 'session_code',
    'difficulty0': 'difficulty0',
    'difficulty1': 'difficulty1',
    'difficulty2': 'difficulty2',
    'difficulty3': 'difficulty3',
    'difficulty4': 'difficulty4',
    'difficulty5': 'difficulty5',
    'difficulty6': 'difficulty6',
    'difficulty7': 'difficulty7',
    'difficulty8': 'difficulty8',
    'difficulty9': 'difficulty9',
    'difficulty10': 'difficulty10',
    'error_difficulty': 'error_difficulty',
    'punish_del_good': 'punish_del_good',
    'punish_del_bad': 'punish_del_bad',
    'punish_nodel_good': 'punish_nodel_good',
    'punish_nodel_bad': 'punish_nodel_bad',
    'switching_point': 'switching_point',
    'research_about': 'research_about',
    'unclear': 'unclear',
    'why_punish': 'why_punish',
    'why_punish_del_less': 'why_punish_del_less',
    'attention': 'attention',
    'Time taken': 'time_taken',
    'Total approvals': 'total_approvals',
    'Ethnicity': 'ethnicity',
    'Gender identity': 'gender',
    'Living abroad': 'abroad',
    'Fluent languages': 'languages',
    'Technology use at work': 'tech_at_work',
    'Employment-sector': 'employ_sector',
    'Leadership/position of power/supervisory duties': 'leadership',
    'Management experience': 'management_exp',
    'Highest education level completed': 'education',
    'Body weight': 'body_weight',
    'Religious affiliation': 'religion',
    'Dating apps': 'dating_apps',
    'Hobbies - categories': 'hobbies',
    'Weekly device usage': 'device_usage',
    'Internet enabled products': 'internet_products',
    'Computer programming': 'programming',
    'Socioeconomic status': 'socio_status',
    'Nft experience': 'nft',
    'Cryptocurrency': 'crypto',
    'Personal income (gbp)': 'income',
    'Age': 'age',
    'Sex': 'sex',
    'Ethnicity simplified': 'ethnicity_simple',
    'Country of birth': 'country_birth',
    'Nationality': 'nationality',
    'Language': 'language',
    'Student status': 'student_status',
    'Employment status': 'employ_status',
    'perception': 'perception',
    'why_punish_bad': 'why_punish_bad',
    'accountability': 'accountability',
    'why_punish_good': 'why_punish_good',
    'responsibility': 'responsibility',
    'trust': 'trust',
    'pilot': 'pilot',
    'treat': 'treat',
    'female': 'female',
    'pass_att2': 'pass_att2',
    'wa_difficulty': 'wa_difficulty',
    'med_difficulty': 'med_difficulty',
    'never_punish': 'never_punish',
    'bad_good_del': 'bad_good_del',
    'bad_good_nodel': 'bad_good_nodel',
    'nodel_del_good': 'nodel_del_good',
    'nodel_del_bad': 'nodel_del_bad',
    'punish_del_good_bin': 'punish_del_good_binary',
    'punish_del_bad_bin': 'punish_del_bad_binary',
    'punish_nodel_good_bin': 'punish_nodel_good_binary',
    'punish_nodel_bad_bin': 'punish_nodel_bad_binary',
    'p_time': 'punish_time',
    'technology_score': 'technology_score',
    'religious': 'religious',
    'went_to_uni': 'went_to_uni',
    'leader': 'leader',
    'tech_use_at_work_often': 'tech_use_at_work_often',
    'white': 'white',
}

ev.rename(columns=rename_dict, inplace=True)
ev = ev[rename_dict.values()]

In [5]:
ev.to_excel('../../processed_data/05_evaluator/evaluator_cleaned.xlsx', index=False)